<a href="https://colab.research.google.com/github/rasia92015-byte/NHANES/blob/main/Notebooks/BChe_IR/BChE_IR_liverfat_from_NHANES_XPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BChE–Insulin Resistance + Liver Fat: Full NHANES 2021–2023 Reproduction

Analysis from raw NHANES XPT files.

**Sections**
1. Create the merged main dataset and derived variables.
2. Run survey-weighted primary, quartile, mediation, RCS, sensitivity, and sex-interaction analyses.
3. Generate Figure 1, Figure 2, Table 1, Table 2, and Supplementary Tables S1–S3.

Per the current analysis plan, this notebook intentionally **does not** run ALT/GGT, triglyceride, or HDL sensitivity analyses.

Required files:
`DEMO_L`, `BCHE_L`, `INS_L`, `GLU_L`, `LUX_L`, `BMX_L`, `SMQ_L`, `ALQ_L`, `PAQ_L`, `DIQ_L`, `GHB_L`.

Change only `DATA_DIR` in the first code cell if needed, then **Runtime → Run all**.


In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
# Put the NHANES XPT files in DATA_DIR (or a subfolder).
# If you upload them directly into the Colab session, /content is fine.
DATA_DIR = "/content"
OUT_DIR = "/content/BChE_IR_results"

import os, glob, math, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display

os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

REQUIRED_XPT = [
    "DEMO_L.xpt",
    "BCHE_L.xpt",
    "INS_L.xpt",
    "GLU_L.xpt",
    "LUX_L.xpt",
    "BMX_L.xpt",
    "SMQ_L.xpt",
    "ALQ_L.xpt",
    "PAQ_L.xpt",
    "DIQ_L.xpt",
    "GHB_L.xpt",
]

def find_file(filename, root=DATA_DIR):
    hits = glob.glob(os.path.join(root, "**", filename), recursive=True)
    if not hits:
        raise FileNotFoundError(f"Missing {filename}. Put it under DATA_DIR={root}")
    if len(hits) > 1:
        print(f"Warning: multiple copies of {filename}; using {hits[0]}")
    return hits[0]

FILES = {f: find_file(f) for f in REQUIRED_XPT}
print("Found all required XPT files:")
for k, v in FILES.items():
    print(f"  {k}: {v}")


In [ ]:
# ============================================================
# 1. CREATE MAIN DATASET FROM RAW XPT FILES
# ============================================================

# pandas.read_sas() can decode SAS/XPORT numeric 0 as the smallest IBM
# floating-point value. Replace only this exact artifact with 0.
SAS_XPORT_TINY_ZERO = 5.397605346934028e-79

def read_xpt(filename, columns):
    path = FILES[filename]
    df = pd.read_sas(path, format="xport", encoding="utf-8")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for c in numeric_cols:
        x = df[c].to_numpy(dtype=float, copy=True)
        mask = np.isclose(x, SAS_XPORT_TINY_ZERO, rtol=0, atol=1e-90)
        if mask.any():
            df.loc[mask, c] = 0.0
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise KeyError(f"{filename} is missing expected columns: {missing}")
    return df[columns].copy()

demo = read_xpt("DEMO_L.xpt", [
    "SEQN","RIDAGEYR","RIAGENDR","RIDRETH3","DMDEDUC2","INDFMPIR",
    "SDMVSTRA","SDMVPSU"
])
bche = read_xpt("BCHE_L.xpt", ["SEQN","LBXBA","LBXCH"])
ins  = read_xpt("INS_L.xpt",  ["SEQN","WTSAF2YR","LBXIN"])
glu  = read_xpt("GLU_L.xpt",  ["SEQN","LBXGLU"])
lux  = read_xpt("LUX_L.xpt",  ["SEQN","LUAXSTAT","LUXCAPM"])
bmx  = read_xpt("BMX_L.xpt",  ["SEQN","BMXBMI","BMXWAIST"])
smq  = read_xpt("SMQ_L.xpt",  ["SEQN","SMQ020","SMQ040"])
alq  = read_xpt("ALQ_L.xpt",  ["SEQN","ALQ111","ALQ121"])
paq  = read_xpt("PAQ_L.xpt",  ["SEQN","PAD790Q","PAD790U","PAD800","PAD810Q","PAD810U","PAD820"])
diq  = read_xpt("DIQ_L.xpt",  ["SEQN","DIQ010","DIQ050","DIQ070"])
ghb  = read_xpt("GHB_L.xpt",  ["SEQN","LBXGH"])

main = demo.copy()
for df in [bche, ins, glu, lux, bmx, smq, alq, paq, diq, ghb]:
    main = main.merge(df, on="SEQN", how="left", validate="one_to_one")

# ---------- Outcomes / exposures ----------
main["HOMAIR"]  = main["LBXIN"] * main["LBXGLU"] / 405.0
main["logHOMA"] = np.log(main["HOMAIR"].where(main["HOMAIR"] > 0))
main["logBA"]    = np.log(main["LBXBA"].where(main["LBXBA"] > 0))
main["logCH"]    = np.log(main["LBXCH"].where(main["LBXCH"] > 0))

# ---------- Education ----------
# 1 = <HS; 2 = HS/GED; 3 = some college/AA; 4 = college graduate+
main["educ4"] = np.select(
    [
        main["DMDEDUC2"].isin([1,2]),
        main["DMDEDUC2"].eq(3),
        main["DMDEDUC2"].eq(4),
        main["DMDEDUC2"].eq(5),
    ],
    [1,2,3,4],
    default=np.nan
)

# ---------- Smoking ----------
# 0 never; 1 former; 2 current
main["smoke3"] = np.nan
main.loc[main["SMQ020"].eq(2), "smoke3"] = 0
main.loc[main["SMQ020"].eq(1) & main["SMQ040"].eq(3), "smoke3"] = 1
main.loc[main["SMQ020"].eq(1) & main["SMQ040"].isin([1,2]), "smoke3"] = 2

# ---------- Alcohol ----------
# 0 never; 1 former/no alcohol in past year; 2 current past-year drinker
main["alcohol3"] = np.nan
main.loc[main["ALQ111"].eq(2), "alcohol3"] = 0
main.loc[main["ALQ111"].eq(1) & main["ALQ121"].eq(0), "alcohol3"] = 1
main.loc[main["ALQ111"].eq(1) & main["ALQ121"].between(1,10), "alcohol3"] = 2

# ---------- Leisure-time physical activity ----------
# PAD790*: moderate; PAD810*: vigorous
# Unit codes are strings: D/W/M/Y.
def weekly_frequency(q, unit):
    out = pd.Series(np.nan, index=q.index, dtype=float)
    out[q.eq(0)] = 0.0
    valid = q.gt(0) & q.lt(7777)
    out.loc[valid & unit.eq("D")] = q.loc[valid & unit.eq("D")] * 7.0
    out.loc[valid & unit.eq("W")] = q.loc[valid & unit.eq("W")]
    out.loc[valid & unit.eq("M")] = q.loc[valid & unit.eq("M")] * (12.0/52.0)
    out.loc[valid & unit.eq("Y")] = q.loc[valid & unit.eq("Y")] * (1.0/52.0)
    return out

mod_freq = weekly_frequency(main["PAD790Q"], main["PAD790U"])
vig_freq = weekly_frequency(main["PAD810Q"], main["PAD810U"])

mod_dur = main["PAD800"].where(main["PAD800"] < 7777)
vig_dur = main["PAD820"].where(main["PAD820"] < 7777)

mod_min = pd.Series(np.where(mod_freq.eq(0), 0.0, mod_freq * mod_dur), index=main.index)
vig_min = pd.Series(np.where(vig_freq.eq(0), 0.0, vig_freq * vig_dur), index=main.index)

valid_mod = mod_freq.notna() & (mod_freq.eq(0) | mod_dur.notna())
valid_vig = vig_freq.notna() & (vig_freq.eq(0) | vig_dur.notna())

main["pa_mmeq_minwk"] = np.where(
    valid_mod & valid_vig,
    mod_min + 2.0 * vig_min,
    np.nan
)

# Three-category PA variable used in Models 2-4:
# 0 = 0 min/wk; 1 = >0 but <150; 2 = >=150 (reference category)
main["pa3"] = np.select(
    [
        main["pa_mmeq_minwk"].eq(0),
        main["pa_mmeq_minwk"].gt(0) & main["pa_mmeq_minwk"].lt(150),
        main["pa_mmeq_minwk"].ge(150),
    ],
    [0,1,2],
    default=np.nan
)

# ---------- Diabetes sensitivity flag ----------
# Exclude if diagnosed diabetes, insulin use, oral diabetes pills,
# fasting glucose >=126 mg/dL, or HbA1c >=6.5%.
main["diabetes_flag"] = (
    main["DIQ010"].eq(1) |
    main["DIQ050"].eq(1) |
    main["DIQ070"].eq(1) |
    main["LBXGLU"].ge(126) |
    main["LBXGH"].ge(6.5)
).astype(float)

# ---------- Core analytic sample ----------
# Adult + fasting subsample + valid FibroScan exam + outcome/exposure/CAP available.
core = main.loc[
    main["RIDAGEYR"].ge(20) &
    main["WTSAF2YR"].gt(0) &
    main["LUAXSTAT"].eq(1) &
    main["logHOMA"].notna() &
    main["logBA"].notna() &
    main["logCH"].notna() &
    main["LUXCAPM"].notna()
].copy()

def weighted_mean_sd(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    ok = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[ok], w[ok]
    mu = np.sum(w*x) / np.sum(w)
    sd = np.sqrt(np.sum(w*(x-mu)**2) / np.sum(w))
    return mu, sd

# Standardize the two log-BChE measures in the same N=2505 core sample.
for raw, zname in [("logBA","z_logBA"), ("logCH","z_logCH")]:
    mu, sd = weighted_mean_sd(core[raw], core["WTSAF2YR"])
    main[zname] = (main[raw] - mu) / sd
    core[zname] = (core[raw] - mu) / sd

print(f"Merged main dataset: N={len(main):,}")
print(f"Core fasting/elastography analytic sample: N={len(core):,}")
assert len(core) == 2505, f"Expected core N=2505, got {len(core)}"

# Save merged datasets.
main.to_csv(os.path.join(OUT_DIR, "BChE_IR_main_dataset.csv.gz"), index=False, compression="gzip")
core.to_csv(os.path.join(OUT_DIR, "BChE_IR_core_N2505.csv"), index=False)
print("Saved main and core datasets.")


In [ ]:
# ============================================================
# 2. ANALYSIS
#    - Survey-weighted Models 1-4
#    - Quartiles / P-trend
#    - CAP statistical mediation
#    - RCS
#    - Sensitivity analyses requested for manuscript
#    - Sex-stratified / interaction analyses
# ============================================================

# ---------- Weighted quantiles ----------
def weighted_quantile(x, w, qs, centered=True):
    """
    centered=True uses midpoint weighted empirical probabilities:
        (cumsum(w) - 0.5*w) / sum(w)
    This is used for descriptive weighted quantiles / medians.
    """
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    qs = np.atleast_1d(qs).astype(float)
    ok = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[ok], w[ok]
    order = np.argsort(x)
    x, w = x[order], w[order]
    if centered:
        p = (np.cumsum(w) - 0.5*w) / np.sum(w)
    else:
        p = np.cumsum(w) / np.sum(w)
    return np.interp(qs, p, x)

# ---------- Design matrix ----------
def covariate_matrix(df, model, include_sex=True, adiposity_var="BMXBMI", include_pir=False):
    X = pd.DataFrame(index=df.index)

    X["age10"] = (df["RIDAGEYR"] - 50.0) / 10.0

    if include_sex:
        X["female"] = (df["RIAGENDR"] == 2).astype(float)

    # Race/ethnicity: reference = non-Hispanic White (RIDRETH3=3)
    for c in [1,2,4,6,7]:
        X[f"race_{c}"] = (df["RIDRETH3"] == c).astype(float)

    if model >= 2:
        # Education: reference = college graduate+ (4)
        for c in [1,2,3]:
            X[f"educ_{c}"] = (df["educ4"] == c).astype(float)

        # Smoking: reference = never (0)
        for c in [1,2]:
            X[f"smoke_{c}"] = (df["smoke3"] == c).astype(float)

        # Alcohol: reference = never (0)
        for c in [1,2]:
            X[f"alcohol_{c}"] = (df["alcohol3"] == c).astype(float)

        # PA: reference = >=150 moderate-equivalent min/wk (2)
        for c in [0,1]:
            X[f"pa_{c}"] = (df["pa3"] == c).astype(float)

        # Preserve missingness of the original categorical variables.
        groups = [
            ("educ4", "educ_"),
            ("smoke3", "smoke_"),
            ("alcohol3", "alcohol_"),
            ("pa3", "pa_"),
        ]
        for raw, prefix in groups:
            miss = df[raw].isna()
            cols = [c for c in X.columns if c.startswith(prefix)]
            X.loc[miss, cols] = np.nan

    if model >= 3:
        X["adiposity"] = df[adiposity_var]

    if model >= 4:
        # Scale CAP only for numerical stability; scaling does not affect
        # exposure coefficients or inference.
        X["CAP10"] = (df["LUXCAPM"] - 250.0) / 10.0

    if include_pir:
        X["PIR"] = df["INDFMPIR"]

    return X

# ---------- Survey-weighted linear regression ----------
def survey_wls(df, y, X, weight="WTSAF2YR"):
    """
    Survey-weighted WLS with Taylor/cluster sandwich variance:
      strata = SDMVSTRA
      PSU    = SDMVPSU
      weight = WTSAF2YR

    The residual score is summed within each PSU, centered within stratum,
    and multiplied by m_h/(m_h-1). Degrees of freedom = PSUs - strata.
    """
    y = pd.Series(y, index=df.index, name="_y")
    tmp = pd.concat(
        [df[[weight, "SDMVSTRA", "SDMVPSU"]], y, X],
        axis=1
    ).dropna()
    tmp = tmp.loc[tmp[weight] > 0].copy()

    xcols = list(X.columns)
    XX = np.column_stack([np.ones(len(tmp)), tmp[xcols].to_numpy(float)])
    yy = tmp["_y"].to_numpy(float)
    ww = tmp[weight].to_numpy(float)

    XtW = XX.T * ww
    bread = XtW @ XX
    bread_inv = np.linalg.pinv(bread)
    beta = bread_inv @ (XtW @ yy)

    resid = yy - XX @ beta
    scores = (ww * resid)[:, None] * XX

    meat = np.zeros((XX.shape[1], XX.shape[1]))
    used_strata = 0
    total_psus = 0

    strata = tmp["SDMVSTRA"].to_numpy()
    psu = tmp["SDMVPSU"].to_numpy()

    for h in np.unique(strata):
        ih = strata == h
        psus_h = np.unique(psu[ih])
        if len(psus_h) < 2:
            continue
        U = np.vstack([
            scores[ih & (psu == p), :].sum(axis=0)
            for p in psus_h
        ])
        Uc = U - U.mean(axis=0)
        meat += len(psus_h)/(len(psus_h)-1) * (Uc.T @ Uc)
        used_strata += 1
        total_psus += len(psus_h)

    cov = bread_inv @ meat @ bread_inv
    se = np.sqrt(np.maximum(np.diag(cov), 0))
    df_design = total_psus - used_strata

    with np.errstate(divide="ignore", invalid="ignore"):
        tstat = beta / se
    p = 2 * stats.t.sf(np.abs(tstat), df=df_design)
    crit = stats.t.ppf(0.975, df=df_design)
    lo = beta - crit * se
    hi = beta + crit * se

    return {
        "n": len(tmp),
        "df": df_design,
        "beta": beta,
        "se": se,
        "p": p,
        "lo": lo,
        "hi": hi,
        "cov": cov,
        "names": ["Intercept"] + xcols,
        "data": tmp,
    }

def effect_row(result, coefficient_index=1):
    b = float(result["beta"][coefficient_index])
    lo = float(result["lo"][coefficient_index])
    hi = float(result["hi"][coefficient_index])
    return {
        "N": result["n"],
        "Beta_log_HOMA": b,
        "Beta_95CI_low": lo,
        "Beta_95CI_high": hi,
        "HOMA_change_pct": 100*np.expm1(b),
        "Pct_CI_low": 100*np.expm1(lo),
        "Pct_CI_high": 100*np.expm1(hi),
        "P": float(result["p"][coefficient_index]),
    }

# ---------- Models 1-4 ----------
model_adjustments = {
    1: "Age + sex + race/ethnicity",
    2: "+ education + smoking + alcohol + leisure-time PA",
    3: "+ BMI",
    4: "+ CAP",
}

model_rows = []
primary_fit = {}

for exp, label in [("z_logBA","BChE activity"), ("z_logCH","BChE concentration")]:
    for model in [1,2,3,4]:
        X = pd.concat([core[[exp]], covariate_matrix(core, model)], axis=1)
        fit = survey_wls(core, core["logHOMA"], X)
        eff = effect_row(fit, 1)
        model_rows.append({
            "Exposure": label,
            "Model": model,
            "Adjustment": model_adjustments[model],
            **eff
        })
        if model == 4:
            primary_fit[exp] = fit

models_df = pd.DataFrame(model_rows)

# Define the common fully adjusted Model 4 sample.
m4cov = covariate_matrix(core, 4)
m4_complete = pd.concat(
    [core[["z_logBA","z_logCH","logHOMA"]], m4cov],
    axis=1
).dropna().index
a4 = core.loc[m4_complete].copy()

print(f"Primary fully adjusted sample: N={len(a4):,}")
assert len(a4) == 2163, f"Expected Model 4 N=2163, got {len(a4)}"

# Reproducibility checkpoints for the two main Model 4 coefficients.
checkpoint = models_df.loc[models_df["Model"].eq(4), ["Exposure","N","Beta_log_HOMA","HOMA_change_pct","P"]]
print("\nPrimary Model 4 checkpoints:")
display(checkpoint)

expected_beta = {
    "BChE activity": 0.0896867953,
    "BChE concentration": 0.0865115475,
}
for label, target in expected_beta.items():
    got = models_df.loc[
        (models_df["Exposure"] == label) & (models_df["Model"] == 4),
        "Beta_log_HOMA"
    ].iloc[0]
    assert np.isclose(got, target, atol=5e-6), (label, got, target)

# ---------- Quartile models ----------
# Use weighted quartiles defined in the final Model 4 sample.
ba_cuts = weighted_quantile(a4["LBXBA"], a4["WTSAF2YR"], [0.25,0.50,0.75], centered=True)
ch_cuts = weighted_quantile(a4["LBXCH"], a4["WTSAF2YR"], [0.25,0.50,0.75], centered=True)

def assign_q(x, cuts):
    q = pd.Series(np.digitize(x, cuts, right=True) + 1, index=x.index, dtype=float)
    q.loc[x.isna()] = np.nan
    return q

a4["ba_q"] = assign_q(a4["LBXBA"], ba_cuts)
a4["ch_q"] = assign_q(a4["LBXCH"], ch_cuts)

def quartile_medians(a4, rawvar, qvar):
    return {
        q: float(weighted_quantile(
            a4.loc[a4[qvar].eq(q), rawvar],
            a4.loc[a4[qvar].eq(q), "WTSAF2YR"],
            [0.5],
            centered=True
        )[0])
        for q in [1,2,3,4]
    }

ba_q_medians = quartile_medians(a4, "LBXBA", "ba_q")
ch_q_medians = quartile_medians(a4, "LBXCH", "ch_q")

def run_quartile_models(rawvar, cuts, medians, label):
    rows = []
    q_all = assign_q(core[rawvar], cuts)

    for model in [1,2,3,4]:
        Q = pd.DataFrame(index=core.index)
        for q in [2,3,4]:
            Q[f"Q{q}"] = (q_all == q).astype(float)
        Q.loc[q_all.isna(), :] = np.nan

        X = pd.concat([Q, covariate_matrix(core, model)], axis=1)
        fit = survey_wls(core, core["logHOMA"], X)

        # Trend score uses the weighted median exposure value within each quartile.
        trend_score = q_all.map(medians).rename("quartile_median_score")
        Xt = pd.concat([trend_score, covariate_matrix(core, model)], axis=1)
        fit_trend = survey_wls(core, core["logHOMA"], Xt)
        p_trend = float(fit_trend["p"][1])

        rows.append({
            "Exposure": label, "Model": model, "N": fit["n"],
            "Quartile": "Q1", "HOMA_change_pct": 0.0,
            "Pct_CI_low": 0.0, "Pct_CI_high": 0.0,
            "P_vs_Q1": np.nan, "P_trend": np.nan
        })

        for i, q in enumerate([2,3,4], start=1):
            b, lo, hi = fit["beta"][i], fit["lo"][i], fit["hi"][i]
            rows.append({
                "Exposure": label, "Model": model, "N": fit["n"],
                "Quartile": f"Q{q}",
                "HOMA_change_pct": 100*np.expm1(b),
                "Pct_CI_low": 100*np.expm1(lo),
                "Pct_CI_high": 100*np.expm1(hi),
                "P_vs_Q1": float(fit["p"][i]),
                "P_trend": p_trend if q == 4 else np.nan
            })
    return pd.DataFrame(rows)

quartile_df = pd.concat([
    run_quartile_models("LBXBA", ba_cuts, ba_q_medians, "BChE activity"),
    run_quartile_models("LBXCH", ch_cuts, ch_q_medians, "BChE concentration"),
], ignore_index=True)

# ---------- CAP statistical mediation ----------
# Same complete N=2163 sample for all paths.
cap_mu, cap_sd = weighted_mean_sd(a4["LUXCAPM"], a4["WTSAF2YR"])
a4["z_CAP"] = (a4["LUXCAPM"] - cap_mu) / cap_sd

def mediation_analysis(df, exp, label):
    m3 = covariate_matrix(df, 3)

    # a path: exposure -> CAP
    fit_a = survey_wls(
        df, df["z_CAP"],
        pd.concat([df[[exp]], m3], axis=1)
    )

    # total c path: exposure -> log(HOMA-IR)
    fit_c = survey_wls(
        df, df["logHOMA"],
        pd.concat([df[[exp]], m3], axis=1)
    )

    # direct c' and b: log(HOMA-IR) ~ exposure + CAP
    fit_b = survey_wls(
        df, df["logHOMA"],
        pd.concat([df[[exp, "z_CAP"]], m3], axis=1)
    )

    a, sea = float(fit_a["beta"][1]), float(fit_a["se"][1])
    b, seb = float(fit_b["beta"][2]), float(fit_b["se"][2])
    c = float(fit_c["beta"][1])
    cp = float(fit_b["beta"][1])

    indirect = a*b
    # Delta-method approximation; separate-equation covariance is set to zero.
    se_indirect = math.sqrt((b*sea)**2 + (a*seb)**2)
    df_indirect = min(fit_a["df"], fit_b["df"])
    crit = stats.t.ppf(0.975, df=df_indirect)
    lo = indirect - crit*se_indirect
    hi = indirect + crit*se_indirect
    p_ind = 2*stats.t.sf(abs(indirect/se_indirect), df=df_indirect)

    return {
        "Exposure": label,
        "N": len(df),
        "Path_a_exposure_to_CAP_beta": a,
        "Path_a_P": float(fit_a["p"][1]),
        "Path_b_CAP_to_logHOMA_beta": b,
        "Path_b_P": float(fit_b["p"][2]),
        "Total_effect_c": c,
        "Total_P": float(fit_c["p"][1]),
        "Direct_effect_c_prime": cp,
        "Direct_P": float(fit_b["p"][1]),
        "Indirect_effect_ab": indirect,
        "Indirect_95CI_low": lo,
        "Indirect_95CI_high": hi,
        "Indirect_P": p_ind,
        "Proportion_statistically_mediated_pct": 100*indirect/c,
    }

mediation_df = pd.DataFrame([
    mediation_analysis(a4, "z_logBA", "BChE activity"),
    mediation_analysis(a4, "z_logCH", "BChE concentration"),
])

print("\nMediation:")
display(mediation_df)

# ---------- Restricted cubic splines ----------
def rcs_basis(x, knots):
    x = np.asarray(x, dtype=float)
    k = np.asarray(knots, dtype=float)
    if len(k) != 4:
        raise ValueError("This notebook uses 4-knot RCS.")
    denom = (k[-1] - k[0])**2
    cols = [x]
    for j in range(len(k)-2):
        kj = k[j]
        term = np.maximum(x-kj, 0)**3
        term -= ((k[-1]-kj)/(k[-1]-k[-2])) * np.maximum(x-k[-2], 0)**3
        term += ((k[-2]-kj)/(k[-1]-k[-2])) * np.maximum(x-k[-1], 0)**3
        cols.append(term / denom)
    return np.column_stack(cols)

def wald_f_p(beta, cov, idx, df_denom):
    idx = list(idx)
    b = np.asarray(beta)[idx]
    V = np.asarray(cov)[np.ix_(idx, idx)]
    q = len(idx)
    stat = float(b.T @ np.linalg.pinv(V) @ b) / q
    return stat, float(stats.f.sf(stat, q, df_denom))

def fit_rcs(df, rawvar, label):
    knots = weighted_quantile(
        df[rawvar], df["WTSAF2YR"],
        [0.05,0.35,0.65,0.95],
        centered=True
    )
    ref = float(weighted_quantile(
        df[rawvar], df["WTSAF2YR"], [0.50], centered=True
    )[0])

    B = rcs_basis(df[rawvar].to_numpy(), knots)
    bnames = ["rcs_linear","rcs_nl1","rcs_nl2"]
    Bdf = pd.DataFrame(B, columns=bnames, index=df.index)

    X = pd.concat([Bdf, covariate_matrix(df, 4)], axis=1)
    fit = survey_wls(df, df["logHOMA"], X)

    # Coefficients: intercept=0, spline terms=1..3.
    _, p_overall = wald_f_p(fit["beta"], fit["cov"], [1,2,3], fit["df"])
    _, p_nonlinear = wald_f_p(fit["beta"], fit["cov"], [2,3], fit["df"])

    low, high = weighted_quantile(
        df[rawvar], df["WTSAF2YR"], [0.01,0.99], centered=True
    )
    grid = np.linspace(low, high, 200)

    Bref = rcs_basis(np.array([ref]), knots)[0]
    Bgrid = rcs_basis(grid, knots)
    D = Bgrid - Bref

    beta_s = fit["beta"][1:4]
    V_s = fit["cov"][1:4, 1:4]

    eta = D @ beta_s
    var = np.einsum("ij,jk,ik->i", D, V_s, D)
    se = np.sqrt(np.maximum(var, 0))
    crit = stats.t.ppf(0.975, fit["df"])

    curve = pd.DataFrame({
        rawvar: grid,
        "pct_diff": 100*np.expm1(eta),
        "lo": 100*np.expm1(eta - crit*se),
        "hi": 100*np.expm1(eta + crit*se),
    })

    summary = {
        "Exposure": label,
        "N": fit["n"],
        "Knots": ", ".join(f"{v:.8g}" for v in knots),
        "Reference": ref,
        "Overall_P": p_overall,
        "Nonlinear_P": p_nonlinear,
    }
    return summary, curve

rcs_activity_summary, rcs_activity = fit_rcs(a4, "LBXBA", "BChE activity")
rcs_conc_summary, rcs_conc = fit_rcs(a4, "LBXCH", "BChE concentration")
rcs_summary_df = pd.DataFrame([rcs_activity_summary, rcs_conc_summary])

print("\nRCS summary:")
display(rcs_summary_df)

# ---------- Sensitivity analyses ----------
def fit_m4(df, exp, adiposity_var="BMXBMI", include_pir=False):
    X = pd.concat([
        df[[exp]],
        covariate_matrix(
            df, 4,
            adiposity_var=adiposity_var,
            include_pir=include_pir
        )
    ], axis=1)
    return survey_wls(df, df["logHOMA"], X)

sens_rows = []
for exp, label in [("z_logBA","BChE activity"), ("z_logCH","BChE concentration")]:
    specs = [
        ("Primary Model 4 (BMI + CAP)", core, "BMXBMI", False),
        ("Exclude diabetes", core.loc[core["diabetes_flag"].eq(0)], "BMXBMI", False),
        ("Waist circumference replaces BMI", core, "BMXWAIST", False),
        ("+ family poverty-income ratio (PIR)", core, "BMXBMI", True),
    ]
    for name, d, adip, pir in specs:
        fit = fit_m4(d, exp, adiposity_var=adip, include_pir=pir)
        eff = effect_row(fit, 1)
        sens_rows.append({
            "Exposure": label,
            "Sensitivity": name,
            **eff
        })

sensitivity_df = pd.DataFrame(sens_rows)

# ---------- Sex-stratified and interaction analyses ----------
sex_rows = []
for exp, label in [("z_logBA","BChE activity"), ("z_logCH","BChE concentration")]:
    # Interaction in the full Model 4 sample.
    base = covariate_matrix(core, 4, include_sex=True)
    female = (core["RIAGENDR"] == 2).astype(float)
    Xi = pd.concat([
        core[[exp]],
        (core[exp] * female).rename("exposure_x_female"),
        base
    ], axis=1)
    fit_i = survey_wls(core, core["logHOMA"], Xi)
    p_interaction = float(fit_i["p"][2])

    for sex_code, sex_label in [(1,"Male"), (2,"Female")]:
        d = core.loc[core["RIAGENDR"].eq(sex_code)].copy()
        X = pd.concat([
            d[[exp]],
            covariate_matrix(d, 4, include_sex=False)
        ], axis=1)
        fit = survey_wls(d, d["logHOMA"], X)
        eff = effect_row(fit, 1)
        sex_rows.append({
            "Exposure": label,
            "Sex": sex_label,
            **eff,
            "P_interaction": p_interaction
        })

sex_df = pd.DataFrame(sex_rows)

# ---------- Display main outputs ----------
print("\nModels 1-4")
display(models_df)

print("\nQuartile models")
display(quartile_df)

print("\nSensitivity analyses")
display(sensitivity_df)

print("\nSex-stratified / interaction")
display(sex_df)


In [ ]:
# ============================================================
# 3. FIGURES + TABLES + EXCEL EXPORT
# ============================================================

# ---------- Table 1 ----------
def weighted_mean(x, w):
    x = np.asarray(x, float)
    w = np.asarray(w, float)
    ok = np.isfinite(x) & np.isfinite(w) & (w > 0)
    return np.sum(x[ok]*w[ok]) / np.sum(w[ok])

def weighted_pct(mask, w):
    mask = np.asarray(mask, float)
    w = np.asarray(w, float)
    ok = np.isfinite(mask) & np.isfinite(w) & (w > 0)
    return 100*np.sum(mask[ok]*w[ok]) / np.sum(w[ok])

def fmt_mean(d, var, digits=1):
    return f"{weighted_mean(d[var], d['WTSAF2YR']):.{digits}f}"

def fmt_median_iqr(d, var, digits=2):
    q = weighted_quantile(
        d[var], d["WTSAF2YR"], [0.25,0.50,0.75], centered=True
    )
    return f"{q[1]:.{digits}f} [{q[0]:.{digits}f}–{q[2]:.{digits}f}]"

def fmt_pct(d, mask, digits=1):
    return f"{weighted_pct(mask, d['WTSAF2YR']):.{digits}f}%"

# Activity quartiles for Table 1.
a4["activity_quartile"] = assign_q(a4["LBXBA"], ba_cuts)

groups = [("Overall", a4)] + [
    (f"Q{q}", a4.loc[a4["activity_quartile"].eq(q)])
    for q in [1,2,3,4]
]

def row_values(func):
    return [func(d) for _, d in groups]

table1_rows = []

table1_rows.append(["Unweighted N"] + [str(len(d)) for _, d in groups])
table1_rows.append(["BChE activity, U/mL"] + row_values(lambda d: fmt_mean(d,"LBXBA",2)))
table1_rows.append(["BChE concentration, ng/mL"] + row_values(lambda d: fmt_mean(d,"LBXCH",1)))
table1_rows.append(["Age, y"] + row_values(lambda d: fmt_mean(d,"RIDAGEYR",1)))
table1_rows.append(["Female"] + row_values(lambda d: fmt_pct(d,d["RIAGENDR"].eq(2))))

race_labels = {
    1:"Mexican American",
    2:"Other Hispanic",
    3:"Non-Hispanic White",
    4:"Non-Hispanic Black",
    6:"Non-Hispanic Asian",
    7:"Other/multiracial",
}
for code, label in race_labels.items():
    table1_rows.append([
        f"Race/ethnicity: {label}"
    ] + row_values(lambda d, c=code: fmt_pct(d,d["RIDRETH3"].eq(c))))

educ_labels = {1:"<High school",2:"High school/GED",3:"Some college/AA",4:"College graduate+"}
for code, label in educ_labels.items():
    table1_rows.append([
        f"Education: {label}"
    ] + row_values(lambda d, c=code: fmt_pct(d,d["educ4"].eq(c))))

smoke_labels = {0:"Never",1:"Former",2:"Current"}
for code, label in smoke_labels.items():
    table1_rows.append([
        f"Smoking: {label}"
    ] + row_values(lambda d, c=code: fmt_pct(d,d["smoke3"].eq(c))))

alcohol_labels = {0:"Never",1:"Former/no past-year",2:"Current"}
for code, label in alcohol_labels.items():
    table1_rows.append([
        f"Alcohol: {label}"
    ] + row_values(lambda d, c=code: fmt_pct(d,d["alcohol3"].eq(c))))

table1_rows.append([
    "Leisure-time PA ≥150 min/wk"
] + row_values(lambda d: fmt_pct(d,d["pa3"].eq(2))))

table1_rows.append(["BMI, kg/m²"] + row_values(lambda d: fmt_mean(d,"BMXBMI",1)))
table1_rows.append(["Waist circumference, cm"] + row_values(lambda d: fmt_mean(d,"BMXWAIST",1)))
table1_rows.append(["CAP, dB/m"] + row_values(lambda d: fmt_mean(d,"LUXCAPM",1)))
table1_rows.append(["HOMA-IR, median [IQR]"] + row_values(lambda d: fmt_median_iqr(d,"HOMAIR",2)))
table1_rows.append(["Fasting insulin, µU/mL, median [IQR]"] + row_values(lambda d: fmt_median_iqr(d,"LBXIN",1)))
table1_rows.append(["Fasting glucose, mg/dL"] + row_values(lambda d: fmt_mean(d,"LBXGLU",1)))
table1_rows.append(["HbA1c, %"] + row_values(lambda d: fmt_mean(d,"LBXGH",2)))

table1_df = pd.DataFrame(
    table1_rows,
    columns=["Characteristic","Overall","Q1","Q2","Q3","Q4"]
)

# ---------- Table 2 ----------
table2_df = models_df.copy()
table2_df["Beta_95CI"] = table2_df.apply(
    lambda r: f"{r['Beta_log_HOMA']:.3f} ({r['Beta_95CI_low']:.3f}–{r['Beta_95CI_high']:.3f})",
    axis=1
)
table2_df["HOMA_pct_95CI"] = table2_df.apply(
    lambda r: f"{r['HOMA_change_pct']:.1f}% ({r['Pct_CI_low']:.1f} to {r['Pct_CI_high']:.1f})",
    axis=1
)
table2_df = table2_df[[
    "Exposure","Model","Adjustment","N","Beta_95CI","HOMA_pct_95CI","P"
]]

# ---------- Supplementary Table S1 ----------
s1_df = quartile_df.copy()

# ---------- Supplementary Table S2 ----------
s2_df = sensitivity_df.copy()

# ---------- Supplementary Table S3 ----------
s3_df = sex_df.copy()

# ---------- Figure 1: two RCS panels ----------
def plot_rcs_panel(curve, xvar, xlabel, summary, panel_letter, path):
    x = curve[xvar].to_numpy(float)
    y = curve["pct_diff"].to_numpy(float)
    lo = curve["lo"].to_numpy(float)
    hi = curve["hi"].to_numpy(float)

    fig, ax = plt.subplots(figsize=(6.4,4.9))
    ax.fill_between(x, lo, hi, alpha=0.20)
    ax.plot(x, y, linewidth=2.2)
    ax.axhline(0, linewidth=1.0)
    ax.axvline(float(summary["Reference"]), linestyle="--", linewidth=1.0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Adjusted difference in HOMA-IR (%)")
    ax.text(
        0.04, 0.96,
        f"Overall P = {summary['Overall_P']:.4f}\n"
        f"Nonlinear P = {summary['Nonlinear_P']:.3f}",
        transform=ax.transAxes, va="top"
    )
    fig.text(0.015, 0.965, panel_letter, fontsize=15, fontweight="bold")
    fig.tight_layout(rect=(0.02,0,1,0.98))
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

f1a = os.path.join(OUT_DIR, "Figure1A_RCS_activity.png")
f1b = os.path.join(OUT_DIR, "Figure1B_RCS_concentration.png")

plot_rcs_panel(
    rcs_activity, "LBXBA", "BChE activity (U/mL)",
    rcs_activity_summary, "A", f1a
)
plot_rcs_panel(
    rcs_conc, "LBXCH", "BChE concentration (ng/mL)",
    rcs_conc_summary, "B", f1b
)

im1 = Image.open(f1a).convert("RGB")
im2 = Image.open(f1b).convert("RGB")
h = max(im1.height, im2.height)
im1 = ImageOps.pad(im1, (im1.width, h), color="white")
im2 = ImageOps.pad(im2, (im2.width, h), color="white")
combined = Image.new("RGB", (im1.width+im2.width, h), "white")
combined.paste(im1, (0,0))
combined.paste(im2, (im1.width,0))
fig1_path = os.path.join(OUT_DIR, "Figure1_RCS_BChE_HOMAIR.png")
combined.save(fig1_path, dpi=(300,300))

# ---------- Figure 2: mediation ----------
fig, ax = plt.subplots(figsize=(9.0,6.2))
ax.set_xlim(0,10)
ax.set_ylim(0,10)
ax.axis("off")

def mediation_panel(y, row, letter):
    exposure = row["Exposure"]
    a = row["Path_a_exposure_to_CAP_beta"]
    pa = row["Path_a_P"]
    b = row["Path_b_CAP_to_logHOMA_beta"]
    pb = row["Path_b_P"]
    c = row["Total_effect_c"]
    cp = row["Direct_effect_c_prime"]
    ind = row["Indirect_effect_ab"]
    lo = row["Indirect_95CI_low"]
    hi = row["Indirect_95CI_high"]
    pind = row["Indirect_P"]
    prop = row["Proportion_statistically_mediated_pct"]
    n = int(row["N"])

    ax.text(0.25,y+1.25,letter,fontsize=15,fontweight="bold")
    ax.text(1.2,y,f"{exposure}\n(1 SD)",ha="center",va="center",
            bbox={"boxstyle":"round,pad=0.35"})
    ax.text(5.0,y+1.15,"CAP",ha="center",va="center",
            bbox={"boxstyle":"round,pad=0.35"})
    ax.text(8.85,y,"log(HOMA-IR)",ha="center",va="center",
            bbox={"boxstyle":"round,pad=0.35"})

    ax.annotate("",xy=(4.15,y+0.98),xytext=(2.05,y+0.18),
                arrowprops={"arrowstyle":"->","linewidth":1.4})
    ax.annotate("",xy=(8.0,y+0.18),xytext=(5.85,y+0.98),
                arrowprops={"arrowstyle":"->","linewidth":1.4})
    ax.annotate("",xy=(8.0,y-0.38),xytext=(2.05,y-0.38),
                arrowprops={"arrowstyle":"->","linewidth":1.4})

    ax.text(3.1,y+1.15,f"a = {a:.3f}\nP = {pa:.2g}",ha="center",fontsize=9)
    ax.text(6.9,y+1.15,f"b = {b:.3f}\nP = {pb:.2g}",ha="center",fontsize=9)
    ax.text(5.0,y-0.72,f"c = {c:.3f}; c′ = {cp:.3f}",ha="center",fontsize=9.5)
    ax.text(
        5.0,y-1.22,
        f"Indirect ab = {ind:.3f} (95% CI {lo:.3f}–{hi:.3f}); "
        f"P = {pind:.2g}; mediated = {prop:.1f}%; N = {n}",
        ha="center",fontsize=8.9
    )

mediation_panel(7.2, mediation_df.iloc[0], "A")
mediation_panel(3.0, mediation_df.iloc[1], "B")
fig.tight_layout()
fig2_path = os.path.join(OUT_DIR, "Figure2_CAP_mediation.png")
fig.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.close(fig)

# ---------- Export all tables ----------
xlsx_path = os.path.join(OUT_DIR, "BChE_IR_recreated_results.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    table1_df.to_excel(writer, sheet_name="Table 1", index=False)
    table2_df.to_excel(writer, sheet_name="Table 2", index=False)
    s1_df.to_excel(writer, sheet_name="Supplementary S1", index=False)
    s2_df.to_excel(writer, sheet_name="Supplementary S2", index=False)
    s3_df.to_excel(writer, sheet_name="Supplementary S3", index=False)
    mediation_df.to_excel(writer, sheet_name="Mediation", index=False)
    rcs_summary_df.to_excel(writer, sheet_name="RCS Summary", index=False)
    rcs_activity.to_excel(writer, sheet_name="RCS Activity Data", index=False)
    rcs_conc.to_excel(writer, sheet_name="RCS Conc Data", index=False)

# Also save individual CSVs for transparent reproducibility.
table1_df.to_csv(os.path.join(OUT_DIR,"Table1.csv"),index=False)
table2_df.to_csv(os.path.join(OUT_DIR,"Table2.csv"),index=False)
s1_df.to_csv(os.path.join(OUT_DIR,"Supplementary_Table_S1.csv"),index=False)
s2_df.to_csv(os.path.join(OUT_DIR,"Supplementary_Table_S2.csv"),index=False)
s3_df.to_csv(os.path.join(OUT_DIR,"Supplementary_Table_S3.csv"),index=False)
mediation_df.to_csv(os.path.join(OUT_DIR,"Mediation.csv"),index=False)
rcs_summary_df.to_csv(os.path.join(OUT_DIR,"RCS_summary.csv"),index=False)

print("\nDONE")
print("Main outputs:")
print(" ", xlsx_path)
print(" ", fig1_path)
print(" ", fig2_path)

print("\nPrimary manuscript results to compare:")
display(table2_df.loc[table2_df["Model"].eq(4)])
display(mediation_df)
display(rcs_summary_df)
